<a href="https://colab.research.google.com/github/DidarulIslamok/AcadGIS/blob/main/Thesis_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.2 MB/s eta 0:00:00


In [2]:
# ============================================================================
# GEE DATA EXTRACTION -- BARIND TRACT ONLY (8 real stations)
# Replaces the old 35-station national extraction.
#
# Fixes vs. the old script:
#   - Scoped to actual Barind Tract stations, not all of Bangladesh
#   - NDVI: MOD13Q1 (250m) with SummaryQA bit masking (drop cloud/snow pixels)
#   - LST: MOD11A1 (1km) with QC_Day bit masking (drop poor-quality retrievals)
#   - ET: MOD16A2GF (gap-filled) for 2000-2022, MOD16A2 for 2023-2024,
#         with ET_QC bit masking -- this is the actual fix for the
#         75%-zero problem, not a downstream patch
#   - Distinguishes "no image available" from "image available but fully
#     masked out by QC" -- the old script only checked collection size
#   - Extended time range to 2000-2024 (full MODIS record) per literature
#     convention, so results are comparable to prior Barind Tract studies
#
# Run this in Python (earthengine-api + geemap), matching your existing
# ee-didarulislam0097 project. Output: three CSVs exported to Google Drive.
# ============================================================================

import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-didarulislam0097')

# ----------------------------------------------------------------------
# 1. STATIONS -- restricted to the actual Barind Tract, not all 35 BMD
# ----------------------------------------------------------------------
barind_stations = [
    {'name': 'Rangpur',     'lat': 25.75, 'lon': 89.25},
    {'name': 'Dinajpur',    'lat': 25.63, 'lon': 88.65},
    {'name': 'Saidpur',     'lat': 25.78, 'lon': 88.92},
    {'name': 'Rajshahi',    'lat': 24.37, 'lon': 88.60},
    {'name': 'Bogura',      'lat': 24.85, 'lon': 89.37},
    {'name': 'Ishwardi',    'lat': 24.13, 'lon': 89.07},
    {'name': 'Badalgachhi', 'lat': 24.90, 'lon': 88.90},
    {'name': 'Tarash',      'lat': 24.35, 'lon': 89.37},
]

# 2km buffer around each station point -- keeps the sensible part of the
# old design (avoids single-pixel noise / urban contamination)
station_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([s['lon'], s['lat']]).buffer(2000), {'station': s['name']})
    for s in barind_stations
])

START_DATE = '2000-01-01'
END_DATE   = '2024-12-31'
GF_CUTOFF  = '2023-01-01'   # MOD16A2GF only covers up to ~2022; switch to MOD16A2 after this

# ----------------------------------------------------------------------
# 2. QC / MASKING HELPERS
# ----------------------------------------------------------------------
def mask_ndvi_summaryqa(img):
    """MOD13Q1 SummaryQA: 0=good, 1=marginal, 2=snow/ice, 3=cloudy.
    Keep only 0 and 1 (good + marginal); drop snow/cloud pixels."""
    qa = img.select('SummaryQA')
    good = qa.lte(1)
    return img.updateMask(good)

def mask_lst_qc(img):
    """MOD11A1 QC_Day bits 0-1: mandatory QA flag. 00 = good quality,
    produced with good quality. Keep only bit pattern 00."""
    qc = img.select('QC_Day')
    mandatory_qa = qc.bitwiseAnd(3)  # bits 0-1
    good = mandatory_qa.eq(0)
    return img.updateMask(good)

def mask_et_qc(img):
    """MOD16A2 / MOD16A2GF ET_QC bit 0: 0 = good quality, 1 = fill value/
    QC problem. This is the actual fix for the zero-inflation bug -- the
    old script never checked this band and averaged in fill/masked values
    as if they were real zero-ET measurements."""
    qc = img.select('ET_QC')
    good = qc.bitwiseAnd(1).eq(0)
    return img.select('ET').updateMask(good)

# ----------------------------------------------------------------------
# 3. NDVI EXTRACTION (MOD13Q1, 250m, 16-day native cadence)
# ----------------------------------------------------------------------
ndvi_coll = (ee.ImageCollection('MODIS/061/MOD13Q1')
             .filterDate(START_DATE, END_DATE)
             .map(mask_ndvi_summaryqa))

def extract_ndvi(image):
    date = image.date()
    return image.select('NDVI').reduceRegions(
        collection=station_fc,
        reducer=ee.Reducer.mean(),
        scale=250
    ).map(lambda f: f.set('date', date.format('YYYY-MM-dd')))

ndvi_data = ndvi_coll.map(extract_ndvi).flatten()

task_ndvi = ee.batch.Export.table.toDrive(
    collection=ndvi_data,
    description='BARIND8_NDVI_QCMASKED',
    fileFormat='CSV',
    folder='GEE_Exports'
)
task_ndvi.start()
print("Task submitted: BARIND8_NDVI_QCMASKED")

# ----------------------------------------------------------------------
# 4. LST EXTRACTION (MOD11A1, 1km, daily -> composited to 16-day windows
#    to align with NDVI's native cadence)
# ----------------------------------------------------------------------
def extract_lst_for_window(image):
    """image here is an NDVI image; we use its date to define a matching
    16-day LST composite window, built only from QC-good daily images."""
    date = image.date()
    lst_window = (ee.ImageCollection('MODIS/061/MOD11A1')
                  .filterDate(date, date.advance(16, 'day'))
                  .map(mask_lst_qc))
    has_data = lst_window.size().gt(0)
    lst_composite = ee.Algorithms.If(
        has_data,
        lst_window.select('LST_Day_1km').mean(),
        ee.Image().rename('LST_Day_1km')  # fully masked, not a fake zero
    )
    return ee.Image(lst_composite).reduceRegions(
        collection=station_fc,
        reducer=ee.Reducer.mean(),
        scale=1000
    ).map(lambda f: f.set('date', date.format('YYYY-MM-dd'),
                           'had_valid_pixels', has_data))

lst_data = ndvi_coll.map(extract_lst_for_window).flatten()

task_lst = ee.batch.Export.table.toDrive(
    collection=lst_data,
    description='BARIND8_LST_QCMASKED',
    fileFormat='CSV',
    folder='GEE_Exports'
)
task_lst.start()
print("Task submitted: BARIND8_LST_QCMASKED")

# ----------------------------------------------------------------------
# 5. ET EXTRACTION -- MOD16A2GF (2000-2022) + MOD16A2 (2023-2024),
#    both QC-masked. This is the actual fix for the 75%-zero problem.
# ----------------------------------------------------------------------
def extract_et_for_window(image, use_gapfilled):
    date = image.date()
    product = 'MODIS/061/MOD16A2GF' if use_gapfilled else 'MODIS/061/MOD16A2'
    et_window = (ee.ImageCollection(product)
                 .filterDate(date, date.advance(16, 'day'))
                 .map(mask_et_qc))
    has_data = et_window.size().gt(0)
    et_composite = ee.Algorithms.If(
        has_data,
        et_window.mean(),
        ee.Image().rename('ET')  # genuinely missing, not a fake zero
    )
    return ee.Image(et_composite).reduceRegions(
        collection=station_fc,
        reducer=ee.Reducer.mean(),
        scale=500
    ).map(lambda f: f.set('date', date.format('YYYY-MM-dd'),
                           'product_used', product,
                           'had_valid_pixels', has_data))

ndvi_pre_gf   = ndvi_coll.filterDate(START_DATE, GF_CUTOFF)
ndvi_post_gf  = ndvi_coll.filterDate(GF_CUTOFF, END_DATE)

et_data_gf   = ndvi_pre_gf.map(lambda img: extract_et_for_window(img, True)).flatten()
et_data_std  = ndvi_post_gf.map(lambda img: extract_et_for_window(img, False)).flatten()
et_data      = et_data_gf.merge(et_data_std)

task_et = ee.batch.Export.table.toDrive(
    collection=et_data,
    description='BARIND8_ET_GAPFILLED_QCMASKED',
    fileFormat='CSV',
    folder='GEE_Exports'
)
task_et.start()
print("Task submitted: BARIND8_ET_GAPFILLED_QCMASKED")

print("\nAll three tasks submitted. Check the GEE Tasks tab in the Code Editor")
print("(or run task.status() here) -- each may take a while given the full")
print("2000-2024 range across 8 stations. Once complete, download the three")
print("CSVs from Google Drive's GEE_Exports folder before running the")
print("corrected analysis pipeline on them.")

Task submitted: BARIND8_NDVI_QCMASKED
Task submitted: BARIND8_LST_QCMASKED
Task submitted: BARIND8_ET_GAPFILLED_QCMASKED

All three tasks submitted. Check the GEE Tasks tab in the Code Editor
(or run task.status() here) -- each may take a while given the full
2000-2024 range across 8 stations. Once complete, download the three
CSVs from Google Drive's GEE_Exports folder before running the
corrected analysis pipeline on them.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

# ফাইলগুলো ড্রাইভের লোকেশন থেকে পড়ুন
ndvi_df = pd.read_csv('/content/drive/MyDrive/GEE_Exports/BARIND8_NDVI_QCMASKED.csv')
lst_df = pd.read_csv('/content/drive/MyDrive/GEE_Exports/BARIND8_LST_QCMASKED.csv')
et_df = pd.read_csv('/content/drive/MyDrive/GEE_Exports/BARIND8_ET_GAPFILLED_QCMASKED.csv')